# Products.csv — Data Exploration

Exploration of `products.csv` following the same structural → null/missingness → duplicate →
value-range checklist used for `reviews.csv`. Covers all 34 columns: structural profile,
nulls, `asin` uniqueness, and column-by-column notes. Referential integrity against
`reviews.csv` is deferred to a joint pass once both files are profiled — see
`docs/schema_design.md`.

## Setup

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
data_path = Path("../data/raw")
products = pd.read_csv(data_path / "products.csv")
products.shape

(728, 34)

## 1. Structural Profile

In [3]:
products.head(1)

,s.no,about_item,asin,availability,best_sellers_rank,brand_name,brand_page_url,breadcrumbs,customer_review_summary,default_variant/0,default_variant/1,default_variant/2,delivery_date,fastest_delivery_date,list_price,manufacturer,model_number,price_value,product_description,product_url,rating_count,rating_distribution/1star,rating_distribution/2star,rating_distribution/3star,rating_distribution/4star,rating_distribution/5star,rating_stars,recent_purchases,scrape_time,seller_name,seller_page_url,title,all_images,rank_1
0,0,Premium Comfort: Crafted from a high-quality c...,B0B59BJG6Y,In Stock,"#56,836 in Clothing, Shoes & Jewelry (See Top ...",MLYENX Store,https://www.amazon.com/stores/MLYENX/page/1FCD...,"Clothing, Shoes & Jewelry › Men › Clothing › A...",Customers find the shirts comfortable and well...,size:Large,"color:5 Pack Black, Dark Grey, Light Blue, Mil...",NaN,"Saturday, March 15","Tomorrow, March 11",List Price: $53.99,NaN,NaN,39.9926,NaN,https://www.amazon.com/dp/B0B59BJG6Y,"1,654 ratings",2%,1%,7%,15%,75%,4.6 out of 5 stars,50+ bought,03-10-2025 21:42,Greenfive,https://www.amazon.com/gp/help/seller/at-a-gla...,4/5 Pack Mens Polo Shirts Short Sleeve Quick D...,['https://m.media-amazon.com/images/I/41yUF65P...,85.0


In [4]:
products_cols = list(products.columns)

## 2. Null / Missingness Pass

In [5]:
for col in products_cols:
    null_pct = round((products[col].isna().sum() / products.shape[0]) * 100, 2)
    print(f"Nulls percent in {col} is: {null_pct}")

Nulls percent in s.no is: 0.0
Nulls percent in about_item is: 0.0
Nulls percent in asin is: 0.0
Nulls percent in availability is: 1.79
Nulls percent in best_sellers_rank is: 23.35
Nulls percent in brand_name is: 0.0
Nulls percent in brand_page_url is: 10.85
Nulls percent in breadcrumbs is: 1.79
Nulls percent in customer_review_summary is: 12.91
Nulls percent in default_variant/0 is: 4.4
Nulls percent in default_variant/1 is: 9.62
Nulls percent in default_variant/2 is: 99.86
Nulls percent in delivery_date is: 3.57
Nulls percent in fastest_delivery_date is: 8.38
Nulls percent in list_price is: 44.92
Nulls percent in manufacturer is: 63.46
Nulls percent in model_number is: 72.66
Nulls percent in price_value is: 2.88
Nulls percent in product_description is: 62.77
Nulls percent in product_url is: 0.0
Nulls percent in rating_count is: 1.24
Nulls percent in rating_distribution/1star is: 0.0
Nulls percent in rating_distribution/2star is: 0.0
Nulls percent in rating_distribution/3star is: 0.0
N

## 3. Duplicate Pass

In [6]:
products["asin"].duplicated().sum()

np.int64(0)

`asin` is fully unique with no duplication — safe key for a `products` collection.

## 4. Column-Specific Notes

### `availability`

In [7]:
products["availability"].value_counts()

availability
In Stock                                       640
Currently unavailable.                          18
In stock                                        16
Only 1 left in stock - order soon.              14
Only 3 left in stock - order soon.              10
Only 5 left in stock - order soon.               6
Only 2 left in stock - order soon.               6
Available to ship in 1-2 days                    2
This item will be released on May 20, 2025.      1
Only 4 left in stock - order soon.               1
Temporarily out of stock.                        1
Name: count, dtype: int64

`availability` has several issues that need unifying:

1. `In Stock` and `In stock` are the same value with a capital letter making the difference
   (noted for the transformation step).
2. Almost all values will need to collapse to one of 2 options — in stock or unavailable —
   or roughly 3 options: in stock, unavailable, and limited.

### `best_sellers_rank`

In [8]:
for rank in products["best_sellers_rank"].dropna().sample(3).values:
    print(rank.split())

['#47,601', 'in', 'Clothing,', 'Shoes', '&', 'Jewelry', '(See', 'Top', '100', 'in', 'Clothing,', 'Shoes', '&', 'Jewelry)', '#3', 'in', "Men's", 'Wool', 'Jackets', '&', 'Coats']
['#12,422', 'in', 'Clothing,', 'Shoes', '&', 'Jewelry', '(See', 'Top', '100', 'in', 'Clothing,', 'Shoes', '&', 'Jewelry)', '#77', 'in', "Men's", 'Casual', 'Pants']
['#7,539', 'in', 'Clothing,', 'Shoes', '&', 'Jewelry', '(See', 'Top', '100', 'in', 'Clothing,', 'Shoes', '&', 'Jewelry)', '#14', 'in', "Men's", 'Athletic', 'Hoodies']


`best_sellers_rank` contains the rank of the product in the best sellers list, and what
matters is the numbers (the ranks). For the transformation step:

1. Take the numbers and provide a new column for the category.
2. Handle the number string → int conversion cleanly.

### `brand_name`

In [9]:
products["brand_name"].sample(5)

242                   Hanes Store
280                TACVASEN Store
553                 PerZeal Store
619               Brand: Rangnavi
651    The Children's Place Store
Name: brand_name, dtype: str

In [10]:
mask = ~products["brand_name"].str.contains("store|brand", case=False, na=True)

single_name = products.loc[mask, "brand_name"].iloc[0]
print(single_name)

Unknown


`brand_name` notes:

1. 79 brand names contain "Store" in them, which will need removing with the pattern
   `"xxx Store"`.
2. The rest contain the word "Brand" at the start, with the pattern `"Brand: xxx"`.
3. One name is literally the string `"Unknown"`, which should be treated as NaN/null —
   to handle in the Transformer.

### `brand_page_url`

In [11]:
urls = products["brand_page_url"]

# Filter for common URL structural anomalies
anomalies = products[
    urls.isna() |
    ~urls.str.startswith(("http://", "https://"), na=False) |
    urls.str.contains(r"\s", na=False) |
    urls.str.contains(r"[<>{}\\|\\^~\[\]`]", na=False)
]

print(f"Found {len(anomalies)} potential anomalies:")
print(anomalies[["brand_name", "brand_page_url"]])

Found 79 potential anomalies:
                 brand_name brand_page_url
22   Brand: U.S. Polo Assn.            NaN
27   Brand: U.S. Polo Assn.            NaN
29   Brand: U.S. Polo Assn.            NaN
31   Brand: U.S. Polo Assn.            NaN
49          Brand: KAOKLRNI            NaN
..                      ...            ...
696           Brand: Gerber            NaN
697           Brand: Gerber            NaN
704          Brand: Generic            NaN
712          Brand: KONQUWA            NaN
723           Brand: QICAMO            NaN

[79 rows x 2 columns]


No suspicious links found, only nulls — safe to use as-is.

### `breadcrumbs`

In [12]:
products["breadcrumbs"]

0      Clothing, Shoes & Jewelry › Men › Clothing › A...
1      Clothing, Shoes & Jewelry › Men › Clothing › S...
2      Clothing, Shoes & Jewelry › Men › Clothing › A...
3      Clothing, Shoes & Jewelry › Men › Clothing › A...
4      Clothing, Shoes & Jewelry › Men › Clothing › A...
                             ...                        
723    Clothing, Shoes & Jewelry › Women › Clothing ›...
724    Clothing, Shoes & Jewelry › Sport Specific Clo...
725    Clothing, Shoes & Jewelry › Women › Clothing ›...
726    Clothing, Shoes & Jewelry › Women › Clothing ›...
727    Clothing, Shoes & Jewelry › Women › Clothing ›...
Name: breadcrumbs, Length: 728, dtype: str

In [13]:
products["breadcrumbs"].loc[
    products["breadcrumbs"].isna() & products["best_sellers_rank"].notna()
]

Series([], Name: breadcrumbs, dtype: str)

Notes:

1. No need to do heavy work on `best_sellers_rank` to extract a category, since
   `breadcrumbs` already provides it with a suitable separator (the `›` trail).
2. When `breadcrumbs` is null, `best_sellers_rank` is also null — but not the other way
   around, which rules out using `breadcrumbs` to compensate for missing
   `best_sellers_rank` values.

### `default_variant/0`, `/1`, `/2`

In [14]:
for col in products_cols:
    if "default_variant" in col:
        print(products[col].sample(5))
        print("=" * 40)

187      size:Large
516      size:Large
533    size:X-Large
580          size:5
511      size:Large
Name: default_variant/0, dtype: str
278                                       NaN
668    color:Blue Swan/Grey Dots/White Floral
599                       color:Black Leather
302                               color:Black
521                               color:Black
Name: default_variant/1, dtype: str
567    NaN
118    NaN
449    NaN
298    NaN
190    NaN
Name: default_variant/2, dtype: str


In [15]:
(~products["default_variant/0"].str.contains("size", case=False, na=True)).sum()

np.int64(11)

In [16]:
# 1. Create the boolean mask for rows that don't contain "size" and aren't NaN
mask = ~products["default_variant/0"].str.contains("size", case=False, na=True)

# 2. Print just the unique variant values to see what they are
print("Values:")
print(products.loc[mask, "default_variant/0"].unique())

# 3. View the full DataFrame for these rows
no_size_df = products[mask]

Values:
<StringArray>
['style:6" Boxer Brief Fly Front With Pouch',                               'color:Black',                         'color:Neon Orange',                               'color:Green',
                             'color:Magenta',                   'color:Jet Black & Beige',                              'color:Maroon',                          'color:Rama Green',
                             'color:Mustard',                          'color:Light Pink']
Length: 10, dtype: str


Most of `default_variant/0` holds the size of the variant, with a small amount of
pollution (11 rows) containing "color" instead — same situation for `default_variant/1`,
mostly color with a small amount of pollution. `default_variant/2` is 99.86% null and not
worth transforming further.

### `delivery_date`

In [17]:
products["delivery_date"].value_counts()

delivery_date
Saturday, March 15     610
March 31 - April 9      12
Monday, March 17         5
March 20 - 21            4
Wednesday, March 12      4
Thursday, March 13       4
April 1 - 10             4
March 14 - 18            3
March 14 - 19            3
March 20 - 26            3
March 19 - 26            2
March 13 - 17            2
March 15 - 19            2
Friday, March 14         2
Tuesday, March 18        2
March 17 - 21            2
April 8 - 18             2
April 9 - 22             2
April 8 - 21             2
March 19 - 28            1
March 18 - 24            1
March 22 - 30            1
March 17 - 22            1
March 21 - 24            1
March 18 - 21            1
March 23 - April 1       1
March 13 - 14            1
March 23 - April 9       1
March 12 - 13            1
March 22 - 28            1
March 17 - 20            1
March 22 - April 9       1
March 17 - 18            1
March 18 - 23            1
March 16 - 19            1
March 15 - 18            1
April 2 - 11  

`delivery_date` notes:

1. The dominant value is `"Saturday, March 15"` (610 rows).
2. The rest (92 rows) are scattered across different formats.
3. The main patterns observed:
   1. `"day_of_week, month day_of_month"` — most rows follow this pattern.
   2. `"month day - day"` — an intra-month range.
   3. `"month day - month day"` — an inter-month range.

### `fastest_delivery_date`

In [18]:
products["fastest_delivery_date"].value_counts()

fastest_delivery_date
Tomorrow, March 11     491
Wednesday, March 12     83
Today                   49
Tomorrow                 7
Thursday, March 13       5
March 17 - 19            2
March 13 - 18            2
March 14 - 19            2
Friday, March 14         2
March 19 - 23            1
March 19 - 25            1
March 22 - 27            1
March 23 - 30            1
Sunday, March 16         1
March 12 - 14            1
March 23 - April 6       1
March 20 - 22            1
March 12 - 13            1
March 22 - 26            1
March 19 - 22            1
March 22 - April 6       1
Monday, March 17         1
March 18 - 19            1
March 16 - 17            1
Saturday, March 15       1
March 14 - 17            1
March 17 - 20            1
March 21 - 31            1
March 21 - 22            1
March 23 - April 2       1
Thursday, March 20       1
March 14 - 18            1
March 18 - 21            1
Name: count, dtype: int64

`fastest_delivery_date` follows the same pattern shape as `delivery_date`.

Worth checking after cleaning both columns: whether `fastest_delivery_date` is always earlier
than or equal to `delivery_date`, or whether any rows contradict that (a data quality check to
run once both are parsed into real dates).

### `customer_review_summary`

In [19]:
for val in products["customer_review_summary"].sample(5).values:
    print(val)
    print("=" * 50)

Customers appreciate the shoes' style, comfort, and value for money. They find the color and design cute and perfect for dresses and pantsuits. Many mention the shoes are soft and cushiony, keeping feet tucked in comfortably without squeaking or squelching. Some customers are pleased with the shoe size, stretchability, and casual wear. However, some have mixed opinions on the fit and quality.
Customers appreciate the dress's style, fit, and fabric quality. They find it pretty, comfortable, and versatile for summer or fall wear. The lightweight material is also appreciated. However, some customers are disappointed with the color.
Customers appreciate the pants' color and material quality. They find the gray color attractive and the pants durable after multiple wears and washings. However, some customers dislike the smell. Opinions vary on sizing, comfort, stretchiness, value for money, and pockets.
Customers appreciate the underpants for their comfort, fit, and quality. They find the so

`customer_review_summary` is free-text, AI/scraper-generated review summaries — 12.91%
null. Values themselves look clean and usable as-is; no structural transformation needed
beyond deciding how to store free text of this length.

### `rating_count`, `rating_distribution/1star`–`5star`, `rating_stars`

In [20]:
products["rating_count"].loc[products["product_url"] == "https://www.amazon.com/dp/B0B59BJG6Y"]

0    1,654 ratings
Name: rating_count, dtype: str

Rating-related columns notes:

1. This data is stale relative to the live site — `rating_count` for this product shows
   `"1,654 ratings"` here, but the actual current count on Amazon is 2,505 (confirmed by
   manually inspecting the product page and comparing).
2. All rating-related columns (`rating_count`, `rating_distribution/*`, `rating_stars`) are
   snapshot values tied to `scrape_time`, not live numbers — they will drift from the real
   site over time. When transforming, this needs one of a few explicit choices: store them as
   a labeled snapshot (paired with `scrape_time`), or drop them if only current data is
   wanted. This is a decision to make deliberately, not by default.
3. All three column groups are string-embedded numbers (`"1,654 ratings"`,
   `"4.6 out of 5 stars"`, `"75%"`) needing extraction to numeric types in the Transformer —
   same shape as `best_sellers_rank`.

In [21]:
products["rating_distribution/1star"].sample(3)

29     4%
487    7%
645    1%
Name: rating_distribution/1star, dtype: str

In [22]:
products["rating_stars"].sample(3)

343    4.6 out of 5 stars
117    4.5 out of 5 stars
440    3.4 out of 5 stars
Name: rating_stars, dtype: str

### `list_price`

In [23]:
list_prices = products["list_price"].dropna().apply(lambda p: float(p.split("$")[1]))
list_prices.describe()

count    401.000000
mean      44.725910
std       28.672587
min        8.000000
25%       24.000000
50%       36.900000
75%       59.990000
max      199.990000
Name: list_price, dtype: float64

In [24]:
products["list_price"].isna().sum() / products.shape[0]

np.float64(0.4491758241758242)

`list_price` notes:

1. Null percentage is fairly high: 44.92%.
2. Numbers (once parsed from the `"List Price: $X"` string): max 199.99, min 8.0, mean
   ~44.73.
3. Keeping nulls as null when transforming is reasonable — no reliable way to compensate a
   missing list price from other columns.

### `manufacturer`

In [25]:
products["manufacturer"].isna().sum() / products.shape[0]

np.float64(0.6346153846153846)

In [26]:
for m in products["manufacturer"].dropna().sample(10).values:
    print(m)

adidas
Wrangler Authentics
KEFITEVD
Under Armour
Jerzees Men's Athletic
Tommy Hilfiger
Under Armour Apparel
Dickies Men's Sportswear
Hanes
Rustler


`manufacturer` notes:

1. 63.46% null.
2. Populated values are clean — no transformation needed on the values themselves.

### `model_number`

In [27]:
products["model_number"].isna().sum() / products.shape[0]

np.float64(0.7266483516483516)

`model_number` is 72.66% null; populated values are mostly scattered SKU-style strings
with no consistent format worth normalizing further.

### `price_value`

In [28]:
products["price_value"].describe()

count    707.000000
mean      35.323948
std       26.422336
min        5.457300
25%       19.990000
50%       28.950000
75%       41.771600
max      249.990000
Name: price_value, dtype: float64

`price_value` is already a clean `float64` column — min 5.46, max 249.99, mean ~35.32,
2.88% null. No coercion needed, unlike `list_price`, which is still string-formatted.

### `product_description`

In [29]:
products["product_description"].dropna().sample(3)

171    Dockers Men's Classic Fit Signature Khaki Lux ...
389    These running shoes are built to help anyone g...
378    Polo Ralph Lauren Men's Ribbed Casual Crew Soc...
Name: product_description, dtype: str

`product_description` is free-text, 62.77% null. Populated values are clean, readable
product copy — no structural issues, just a high null rate to account for.

### `product_url`

In [30]:
products["product_url"].isna().sum(), products["product_url"].duplicated().sum()

(np.int64(0), np.int64(0))

`product_url` has zero nulls and zero duplicates — one clean URL per product.

### `recent_purchases`

In [31]:
products["recent_purchases"].value_counts().head(10)

recent_purchases
100+ bought    108
200+ bought     76
1K+ bought      63
300+ bought     59
50+ bought      52
400+ bought     52
500+ bought     38
2K+ bought      26
700+ bought     22
600+ bought     21
Name: count, dtype: int64

`recent_purchases` values follow a consistent `"<N>+ bought"` pattern (`"100+ bought"`,
`"1K+ bought"`, etc.) — will need parsing to extract the numeric floor, including handling the
`K` suffix as thousands. 21.84% null.

### `scrape_time`

In [32]:
products["scrape_time"].dropna().sample(5)

190    03-10-2025 21:54
264    03-10-2025 21:59
404    03-10-2025 22:08
136    03-10-2025 21:51
503    03-10-2025 22:39
Name: scrape_time, dtype: str

`scrape_time` is a consistent `MM-DD-YYYY HH:MM` string, all from the same scraping run
(March 10, 2025) — no nulls, straightforward to parse to a real datetime. This is the field
that should anchor any of the stale rating/pricing values noted above.

### `seller_name`, `seller_page_url`

In [33]:
products["seller_name"].isna().sum() / products.shape[0]

np.float64(0.028846153846153848)

In [34]:
products["seller_name"].dropna().sample(5)

145         Amazon.com
543          lionstill
279    COOFANDY Online
660         Amazon.com
551             CUPSHE
Name: seller_name, dtype: str

`seller_name` is 2.88% null, values look clean (store names or `"Amazon.com"` directly).
`seller_page_url` is 41.48% null — high, but expected, since not every seller has a
storefront page (e.g. when Amazon itself is the seller).

### `title`

In [35]:
products["title"].sample(3)

670    Fullfamous Baby Girl's 3pc Rib Frill Long Slee...
220    Amazon Essentials Men's Performance Tech Loose...
363    Men's Disposable Underwear and socks, 14 Pack ...
Name: title, dtype: str

`title` has zero nulls — full-length, descriptive product titles, no structural issues.

### `all_images`

In [36]:
products["all_images"].dropna().iloc[0][:300]

"['https://m.media-amazon.com/images/I/41yUF65PmbL.jpg', 'https://m.media-amazon.com/images/I/41jMweQcgzL.jpg', 'https://m.media-amazon.com/images/I/41QOPMJZtSL.jpg', 'https://m.media-amazon.com/images/I/5153qm1UTML.jpg', 'https://m.media-amazon.com/images/I/41JBALVmKiL.jpg', 'https://m.media-amazon."

`all_images` stores a Python-list-formatted string of image URLs
(`"['https://...', 'https://...']"`), not real JSON — will need `ast.literal_eval` or similar
in the Transformer to parse into an actual list rather than `json.loads` (which would fail on
single-quoted strings). Zero nulls.

### `rank_1`

In [37]:
products["rank_1"].describe()

count    540.000000
mean     174.622222
std      240.115097
min        1.000000
25%       18.000000
50%       56.500000
75%      220.250000
max      994.000000
Name: rank_1, dtype: float64

`rank_1` is already a clean numeric column (float64) — min 1.0, max 994.0, mean ~174.6,
25.82% null. This appears to be the numeric sub-category rank already extracted from
`best_sellers_rank`'s second number (e.g. the `#2` in `"#2 in Men's Jeans"`) — worth
confirming this relationship directly once `best_sellers_rank` is parsed in the Transformer,
since it may make separately re-parsing that number from the string redundant.

## Summary

All 34 columns profiled. Key patterns found across the file:

- **String-embedded numbers** needing extraction in the Transformer: `best_sellers_rank`,
  `rating_count`, `rating_distribution/1star`–`5star`, `rating_stars`, `list_price`,
  `recent_purchases`.
- **Already-clean numeric columns**, no coercion needed: `price_value`, `rank_1`.
- **Free-text fields**, high null rate but clean values: `product_description`,
  `customer_review_summary`, `manufacturer`.
- **Structured-but-inconsistent fields** needing parsing logic: `default_variant/0-2`,
  `delivery_date`, `fastest_delivery_date`, `all_images` (stringified list, not real JSON).
- **Stale/snapshot data**: all rating-related columns are tied to `scrape_time` and do not
  reflect current live values — confirmed via manual cross-check against the live product
  page. This needs an explicit handling decision, not a default one.
- **Clean, no-issue columns**: `asin`, `product_url`, `title`, `brand_page_url`.

Referential integrity against `reviews.csv` (orphan `asin` values, reviews-per-product
cardinality) is covered separately in `docs/schema_design.md`, once both files are loaded
together.